# Universe Filter Playground
Tier 1 weekly filter: large-cap US stocks → `data/watchlist.csv`

In [1]:
# Cell 1 — Setup & import
import sys
sys.path.insert(0, 'backend/01_scanner')

from universe_filter import (
    get_max_share_price,
    get_earnings_tickers,
    run_universe_filter,
    save_watchlist,
)
import pandas as pd


In [2]:
# Cell 2 — Happy path: run full filter and inspect output
count = run_universe_filter()
print(f'\nResult: {count} stocks saved')

if count:
    df = pd.read_csv('data/watchlist.csv')
    display(df.head(10))
    print(f'\nColumns: {df.columns.tolist()}')
    print(f'Price range: ${df["price"].min():.2f} – ${df["price"].max():.2f}')
    print(f'ATR% range:  {df["atr_pct"].min():.1f}% – {df["atr_pct"].max():.1f}%')

[universe] Alpaca unavailable, using fallback ceiling $360.00: Alpaca credentials not set
[universe] price ceiling: $360.00
[universe] querying TradingView screener...
[universe] 65 stocks after ATR% filter (1.0%–5.0%)
[universe] saved 65 tickers to data/watchlist.csv

Result: 65 stocks saved


,ticker,price,volume,atr,atr_pct,rsi,sma20,sma50,sector
0,NVDA,192.53,151446724,7.29,3.79,37.5,207.76,210.09,Electronic Technology
1,AMZN,232.69,77394835,7.94,3.41,39.5,244.03,256.13,Retail Trade
2,AAPL,283.78,77209808,8.16,2.87,41.2,298.29,291.50,Electronic Technology
3,T,22.72,61335176,0.58,2.57,41.8,23.07,24.52,Communications
4,NFLX,73.81,57967057,2.46,3.34,31.5,79.25,86.03,Technology Services
5,PFE,24.29,57453597,0.58,2.38,35.7,25.44,25.97,Health Technology
6,BAC,57.88,39048487,1.19,2.05,71.0,55.21,53.31,Finance
7,VZ,46.54,28996832,1.08,2.31,49.4,46.45,46.96,Communications
8,GOOG,334.69,28486391,11.41,3.41,33.4,358.71,366.63,Technology Services
9,CSCO,113.77,27192286,4.03,3.55,44.9,121.29,108.69,Electronic Technology



Columns: ['ticker', 'price', 'volume', 'atr', 'atr_pct', 'rsi', 'sma20', 'sma50', 'sector']
Price range: $22.72 – $358.33
ATR% range:  1.7% – 4.8%


In [3]:
# Cell 3 — Parameter variations

# Vary earnings window: how many stocks get removed at different lookforward windows?
for days in [3, 5, 10]:
    tickers = get_earnings_tickers(days_ahead=days)
    count = len(tickers) if tickers else 0
    print(f'days_ahead={days}: {count} earnings events')

# Show current price ceiling
print(f'\nCurrent price ceiling: ${get_max_share_price():.2f}')

days_ahead=3: 52 earnings events
days_ahead=5: 53 earnings events
days_ahead=10: 120 earnings events
[universe] Alpaca unavailable, using fallback ceiling $360.00: Alpaca credentials not set

Current price ceiling: $360.00


In [ ]:
# Cell 4 — Parameter variations: inject max_price + cap screener rows
# max_price bypasses the Alpaca lookup (reproducible); limit shrinks the query.
# run_universe_filter always writes watchlist.csv, so snapshot + restore it to avoid
# clobbering the canonical Tier-1 list produced by Cell 2.
import shutil

shutil.copy('data/watchlist.csv', 'data/watchlist.bak.csv')
count = run_universe_filter(max_price=250.0, limit=50)
print(f'\nResult with max_price=$250, limit=50: {count} stocks saved')

if count:
    df = pd.read_csv('data/watchlist.csv')
    print(f'Max price in watchlist: ${df["price"].max():.2f}  (expected <= $250)')

shutil.move('data/watchlist.bak.csv', 'data/watchlist.csv')
print('restored canonical watchlist.csv')

In [4]:
# Cell 4 — Failure path: confirm graceful degradation
import os

# Temporarily remove Finnhub key — should skip earnings filter, not crash
original_key = os.environ.pop('FINNHUB_API_KEY', None)
result = get_earnings_tickers()
print(f'Missing Finnhub key → returned: {result}  (expected: None)')

# Restore key
if original_key:
    os.environ['FINNHUB_API_KEY'] = original_key

# Bad URL in earnings (simulate network failure)
import unittest.mock as mock
with mock.patch('requests.get', side_effect=Exception('network error')):
    result = get_earnings_tickers()
    print(f'Network failure → returned: {result}  (expected: None)')

[universe] FINNHUB_API_KEY not set, skipping earnings filter
Missing Finnhub key → returned: None  (expected: None)
[universe] Finnhub unavailable, skipping earnings filter: network error
Network failure → returned: None  (expected: None)


In [ ]:
# Cell 5 — Free play
